In [1]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
from datasets import load_dataset
from tqdm.auto import tqdm
import math
import gc  # Import the garbage collection module

# --- Prerequisites ---
# For Flash Attention 2, you need to install the library and have a compatible GPU (NVIDIA Ampere or newer).
# Run: pip install flash-attn --no-build-isolation

# --- 1. Configuration ---
model_name = "Qwen/Qwen2-1.5B-Instruct"
dataset_name = "wikitext"
dataset_config = "wikitext-2-raw-v1"

# Training Hyperparameters
num_epochs = 10
batch_size = (
    2  # Keep low to fit on consumer GPUs. Can be increased with Flash Attention.
)
max_seq_length = 256
learning_rate = 5e-6
# Use a smaller subset for a quicker demonstration
train_set_size = 1000
validation_set_size = 200

# Metric Learning Hyperparameters
triplet_loss_margin = 0.05  # The "buffer zone" size
triplet_loss_alpha = 0.1  # How much weight to give the margin loss

# --- 2. Load Tokenizer and Prepare Datasets ---
print("--- Loading Tokenizer and Preparing Datasets ---")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

raw_datasets = load_dataset(dataset_name, dataset_config)


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        return_tensors="pt",
    )


tokenized_datasets = raw_datasets.map(
    tokenize_function, batched=True, remove_columns=["text"]
)
tokenized_datasets.set_format("torch")

train_dataset = (
    tokenized_datasets["train"].shuffle(seed=42).select(range(train_set_size))
)
valid_dataset = (
    tokenized_datasets["validation"].shuffle(seed=42).select(range(validation_set_size))
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)


# --- Helper Function for Robust Model Loading ---
def load_model_with_flash_attention(model_name, device):
    """Loads a model, attempting to use Flash Attention 2 with a fallback."""
    attn_implementation = "flash_attention_2"
    print(f"\n--- Loading model: '{model_name}' ---")
    try:
        print(f"Attempting to load with attn_implementation='{attn_implementation}'...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            attn_implementation=attn_implementation,
        )
        print("Successfully loaded model with Flash Attention 2.")
    except Exception as e:
        print(f"\nWARNING: Failed to load with '{attn_implementation}'. Error: {e}")
        print("Falling back to 'sdpa' (PyTorch's native Scaled Dot Product Attention).")
        print(
            "For Flash Attention 2, ensure you have a compatible GPU (NVIDIA Ampere+) and run: pip install flash-attn --no-build-isolation\n"
        )
        attn_implementation = "sdpa"
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            attn_implementation=attn_implementation,
        )
    model.to(device)
    return model


# --- 3. Evaluation Function ---
def evaluate(model, dataloader):
    model.eval()
    total_eval_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = input_ids.clone()
            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            loss = outputs.loss
            total_eval_loss += loss.item()

    avg_loss = total_eval_loss / len(dataloader)
    perplexity = math.exp(avg_loss)
    model.train()  # Set back to training mode
    return avg_loss, perplexity


# --- 4. Experiment 1: Baseline Fine-Tuning (Cross-Entropy Only) ---
print("\n--- Experiment 1: Training Baseline Model (CE Loss Only) ---")
baseline_model = load_model_with_flash_attention(model_name, device)

optimizer = AdamW(baseline_model.parameters(), lr=learning_rate)
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
progress_bar = tqdm(range(num_training_steps), desc="Baseline Training")

baseline_model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = input_ids.clone()
        outputs = baseline_model(
            input_ids=input_ids, attention_mask=attention_mask, labels=labels
        )
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
        progress_bar.set_description(f"Baseline CE Loss: {loss.item():.4f}")

print("\n--- Evaluating Baseline Model ---")
baseline_val_loss, baseline_perplexity = evaluate(baseline_model, valid_dataloader)


# --- AGGRESSIVE MEMORY CLEANUP ---
print("\n--- Clearing memory before the next experiment ---")
baseline_model.to("cpu")  # Move model to CPU to free GPU memory
del baseline_model
del optimizer
del lr_scheduler
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")
# -----------------------------------


# --- 5. Experiment 2: Fine-Tuning with Compound Loss ---
print("\n\n--- Experiment 2: Training Compound Loss Model (CE + Triplet) ---")
compound_model = load_model_with_flash_attention(model_name, device)

optimizer = AdamW(compound_model.parameters(), lr=learning_rate)
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
progress_bar = tqdm(range(num_training_steps), desc="Compound Training")

compound_model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = input_ids.clone()
        outputs = compound_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,
        )

        # a) Standard Cross-Entropy Loss
        loss_ce = outputs.loss

        # b) Vectorized Triplet Margin Loss
        logits = outputs.logits
        hidden_states = outputs.hidden_states[-1][:, :-1, :]
        embedding_matrix = compound_model.get_output_embeddings().weight
        anchor_h = hidden_states
        positive_ids = labels[:, 1:]
        positive_vecs = embedding_matrix[positive_ids]
        next_token_logits = logits[:, :-1, :]
        one_hot_labels = F.one_hot(positive_ids, num_classes=logits.size(-1)).bool()
        masked_logits = next_token_logits.masked_fill(one_hot_labels, -float("inf"))
        hard_negative_ids = torch.argmax(masked_logits, dim=-1)
        negative_vecs = embedding_matrix[hard_negative_ids]
        score_p = torch.einsum("bsh,bsh->bs", anchor_h, positive_vecs)
        score_n = torch.einsum("bsh,bsh->bs", anchor_h, negative_vecs)
        triplet_loss = F.relu(triplet_loss_margin - (score_p - score_n))
        loss_mask = (attention_mask[:, :-1] == 1) & (
            labels[:, 1:] != tokenizer.pad_token_id
        )
        masked_triplet_loss = triplet_loss * loss_mask
        loss_triplet = masked_triplet_loss.sum() / (loss_mask.sum() + 1e-9)

        # c) Combine the Losses
        total_loss = loss_ce + triplet_loss_alpha * loss_triplet
        total_loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
        progress_bar.set_description(
            f"Total: {total_loss.item():.3f}, CE: {loss_ce.item():.3f}, Triplet: {loss_triplet.item():.3f}"
        )

print("\n--- Evaluating Compound Loss Model ---")
compound_val_loss, compound_perplexity = evaluate(compound_model, valid_dataloader)

# --- 6. Final Comparison ---
print("\n\n--- Experiment Results ---")
print(
    f"Baseline Model      -> Validation Loss: {baseline_val_loss:.4f}, Perplexity: {baseline_perplexity:.4f}"
)
print(
    f"Compound Loss Model -> Validation Loss: {compound_val_loss:.4f}, Perplexity: {compound_perplexity:.4f}"
)

if compound_val_loss < baseline_val_loss:
    print(
        "\nConclusion: The compound loss model performed BETTER on the validation set."
    )
else:
    print(
        "\nConclusion: The baseline (CE only) model performed BETTER on the validation set."
    )

--- Loading Tokenizer and Preparing Datasets ---
Using device: cuda

--- Experiment 1: Training Baseline Model (CE Loss Only) ---

--- Loading model: 'Qwen/Qwen2-1.5B-Instruct' ---
Attempting to load with attn_implementation='flash_attention_2'...


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Successfully loaded model with Flash Attention 2.


Baseline Training:   0%|          | 0/5000 [00:00<?, ?it/s]


--- Evaluating Baseline Model ---


...


--- Clearing memory before the next experiment ---
Memory cleared.


--- Experiment 2: Training Compound Loss Model (CE + Triplet) ---

--- Loading model: 'Qwen/Qwen2-1.5B-Instruct' ---
Attempting to load with attn_implementation='flash_attention_2'...
Successfully loaded model with Flash Attention 2.


Compound Training:   0%|          | 0/5000 [00:00<?, ?it/s]


--- Evaluating Compound Loss Model ---


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]



--- Experiment Results ---
Baseline Model      -> Validation Loss: 0.7215, Perplexity: 2.0575
Compound Loss Model -> Validation Loss: 0.7402, Perplexity: 2.0964

Conclusion: The baseline (CE only) model performed BETTER on the validation set.


### Multi-Margin loss

In [1]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler
from datasets import load_dataset
from tqdm.auto import tqdm
import math
import gc

# --- Prerequisites ---
# For Flash Attention 2, you need to install the library and have a compatible GPU (NVIDIA Ampere or newer).
# Run: pip install flash-attn --no-build-isolation

# --- 1. Configuration ---
model_name = "Qwen/Qwen2-1.5B-Instruct"
dataset_name = "wikitext"
dataset_config = "wikitext-2-raw-v1"

# Training Hyperparameters
num_epochs = 10
batch_size = 2
max_seq_length = 256
learning_rate = 5e-6
train_set_size = 1000
validation_set_size = 200

# NEW: Multi-Margin Loss Hyperparameters
mm_loss_margin = 0.05  # The "buffer zone" size, same concept as before
mm_loss_alpha = 0.1  # How much weight to give the multi-margin loss

# --- 2. Load Tokenizer and Prepare Datasets ---
print("--- Loading Tokenizer and Preparing Datasets ---")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

raw_datasets = load_dataset(dataset_name, dataset_config)


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        return_tensors="pt",
    )


tokenized_datasets = raw_datasets.map(
    tokenize_function, batched=True, remove_columns=["text"]
)
tokenized_datasets.set_format("torch")

train_dataset = (
    tokenized_datasets["train"].shuffle(seed=42).select(range(train_set_size))
)
valid_dataset = (
    tokenized_datasets["validation"].shuffle(seed=42).select(range(validation_set_size))
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)


# --- Helper Function for Robust Model Loading ---
def load_model_with_flash_attention(model_name, device):
    """Loads a model, attempting to use Flash Attention 2 with a fallback."""
    attn_implementation = "flash_attention_2"
    print(f"\n--- Loading model: '{model_name}' ---")
    try:
        print(f"Attempting to load with attn_implementation='{attn_implementation}'...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            attn_implementation=attn_implementation,
        )
        print("Successfully loaded model with Flash Attention 2.")
    except Exception as e:
        print(f"\nWARNING: Failed to load with '{attn_implementation}'. Error: {e}")
        print("Falling back to 'sdpa' (PyTorch's native Scaled Dot Product Attention).")
        attn_implementation = "sdpa"
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            attn_implementation=attn_implementation,
        )
    model.to(device)
    return model


# --- 3. Evaluation Function ---
def evaluate(model, dataloader):
    model.eval()
    total_eval_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = input_ids.clone()
            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            total_eval_loss += outputs.loss.item()
    avg_loss = total_eval_loss / len(dataloader)
    perplexity = math.exp(avg_loss)
    model.train()
    return avg_loss, perplexity


# --- 4. Experiment 1: Baseline Fine-Tuning (Cross-Entropy Only) ---
print("\n--- Experiment 1: Training Baseline Model (CE Loss Only) ---")
baseline_model = load_model_with_flash_attention(model_name, device)
optimizer = AdamW(baseline_model.parameters(), lr=learning_rate)
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
progress_bar = tqdm(range(num_training_steps), desc="Baseline Training")

baseline_model.train()
# for epoch in range(num_epochs):
#     for batch in train_dataloader:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = input_ids.clone()
#         outputs = baseline_model(
#             input_ids=input_ids, attention_mask=attention_mask, labels=labels
#         )
#         loss = outputs.loss
#         loss.backward()
#         optimizer.step()
#         lr_scheduler.step()
#         optimizer.zero_grad()
#         progress_bar.update(1)
#         progress_bar.set_description(f"Baseline CE Loss: {loss.item():.4f}")

print("\n--- Evaluating Baseline Model ---")
baseline_val_loss, baseline_perplexity = evaluate(baseline_model, valid_dataloader)
print("\n" + "=" * 50)
print("--- INTERMEDIATE RESULT: BASELINE MODEL PERFORMANCE ---")
print(
    f"Baseline Model      -> Validation Loss: {baseline_val_loss:.4f}, Perplexity: {baseline_perplexity:.4f}"
)
print("=" * 50)

print("\n--- Clearing memory before the next experiment ---")
baseline_model.to("cpu")
del baseline_model, optimizer, lr_scheduler
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

# --- 5. Experiment 2: Fine-Tuning with Compound Loss (CE + Multi-Margin) ---
print("\n\n--- Experiment 2: Training Compound Loss Model (CE + Multi-Margin) ---")
compound_model = load_model_with_flash_attention(model_name, device)
optimizer = AdamW(compound_model.parameters(), lr=learning_rate)
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
progress_bar = tqdm(range(num_training_steps), desc="Compound Training")

compound_model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = input_ids.clone()
        outputs = compound_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,
        )

        # a) Standard Cross-Entropy Loss
        loss_ce = outputs.loss

        # b) Vectorized Multi-Margin Loss
        logits = outputs.logits[:, :-1, :]  # Scores for all tokens at each position
        positive_ids = labels[:, 1:]

        # Get the score of the single correct token at each position
        # shape: (batch_size, seq_len-1)
        score_p = logits.gather(2, positive_ids.unsqueeze(2)).squeeze(2)

        # Calculate the margin loss term for ALL tokens in the vocabulary
        # We want: margin - (score_p - score_n) for every n
        # This can be broadcast: shape (b, s, 1) - (b, s, v) -> (b, s, v)
        loss_term = mm_loss_margin - (score_p.unsqueeze(2) - logits)

        # We only want to penalize tokens where the loss is positive
        # This is our set of "confusing negatives"
        potential_losses = F.relu(loss_term)

        # CRITICAL: We must NOT penalize the correct (positive) token.
        # So we set its potential loss to 0.
        one_hot_labels = F.one_hot(positive_ids, num_classes=logits.size(-1))
        # potential_losses.masked_fill_(one_hot_labels.bool(), 0)
        potential_losses = potential_losses.masked_fill(one_hot_labels.bool(), 0)

        # Sum the penalties for all confusing negatives at each position
        # shape: (batch_size, seq_len-1)
        summed_loss_per_position = potential_losses.sum(dim=-1)

        # Mask out padding positions from the loss calculation
        loss_mask = (attention_mask[:, :-1] == 1) & (
            labels[:, 1:] != tokenizer.pad_token_id
        )
        masked_total_loss = summed_loss_per_position * loss_mask

        # Average the loss over only the valid, non-padded tokens in the batch
        loss_mm = masked_total_loss.sum() / (loss_mask.sum() + 1e-9)

        # c) Combine the Losses
        total_loss = loss_ce + mm_loss_alpha * loss_mm
        total_loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
        progress_bar.set_description(
            f"Total: {total_loss.item():.3f}, CE: {loss_ce.item():.3f}, MM_Loss: {loss_mm.item():.3f}"
        )

print("\n--- Evaluating Compound Loss Model ---")
compound_val_loss, compound_perplexity = evaluate(compound_model, valid_dataloader)

# --- 6. Final Comparison ---
print("\n\n" + "=" * 50)
print("--- FINAL EXPERIMENT RESULTS ---")
print(
    f"Baseline Model      -> Validation Loss: {baseline_val_loss:.4f}, Perplexity: {baseline_perplexity:.4f}"
)
print(
    f"Compound Loss Model -> Validation Loss: {compound_val_loss:.4f}, Perplexity: {compound_perplexity:.4f}"
)
print("=" * 50)

if compound_val_loss < baseline_val_loss:
    print(
        "\nConclusion: The compound loss model performed BETTER on the validation set."
    )
else:
    print(
        "\nConclusion: The baseline (CE only) model performed BETTER on the validation set."
    )

--- Loading Tokenizer and Preparing Datasets ---
Using device: cuda

--- Experiment 1: Training Baseline Model (CE Loss Only) ---

--- Loading model: 'Qwen/Qwen2-1.5B-Instruct' ---
Attempting to load with attn_implementation='flash_attention_2'...


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Successfully loaded model with Flash Attention 2.


Baseline Training:   0%|          | 0/5000 [00:00<?, ?it/s]


--- Evaluating Baseline Model ---


...


--- INTERMEDIATE RESULT: BASELINE MODEL PERFORMANCE ---
Baseline Model      -> Validation Loss: 9.0645, Perplexity: 8642.6862

--- Clearing memory before the next experiment ---
Memory cleared.


--- Experiment 2: Training Compound Loss Model (CE + Multi-Margin) ---

--- Loading model: 'Qwen/Qwen2-1.5B-Instruct' ---
Attempting to load with attn_implementation='flash_attention_2'...
Successfully loaded model with Flash Attention 2.


Compound Training:   0%|          | 0/5000 [00:00<?, ?it/s]


--- Evaluating Compound Loss Model ---


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]



--- FINAL EXPERIMENT RESULTS ---
Baseline Model      -> Validation Loss: 9.0645, Perplexity: 8642.6862
Compound Loss Model -> Validation Loss: 0.7731, Perplexity: 2.1665

Conclusion: The compound loss model performed BETTER on the validation set.


In [ ]:
# Baseline Model      -> Validation Loss: 0.7240, Perplexity: 2.0626
# Compound Loss Model -> Validation Loss: 0.7731, Perplexity: 2.1665